In [2]:
import pandas as pd
import numpy as np
import xarray as xr
import glob
import yaml
import os
import matplotlib.pyplot as plt
import pypsa

path = '/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten'

In [3]:
## Turbine in use: Vestas V112 (3 MW)
name = 'Vestas_V112_3MW'

# Load turbine configuration (power curve, wind speed bins, rated power, hub height)
with open(str(path + '/resources/' + name + '.yaml')) as f:
    config = yaml.safe_load(f)

# Extract arrays/parameters from config
POW, V, P, HUB_HEIGHT = (
    config["POW"],        # turbine power at each wind speed (e.g., in W or kW)
    config["V"],          # wind speed bins corresponding to the power curve (m/s)
    config["P"],          # rated (nameplate) power (same unit as POW)
    config["HUB_HEIGHT"]  # hub height of the turbine (m)
)

# HUB_HEIGHT = 100  # (optional override if desired)

# Convert lists to NumPy arrays for fast vectorized operations
POW = np.array(POW)
V   = np.array(V)


def average_lat_long(ds, to_height=HUB_HEIGHT, from_height=10, wnd_shear_exp=0.143):
    """
    Compute an area-mean wind speed over a latitude/longitude box and scale it
    from the reference height (10 m) to the turbine hub height via the power law.

    Parameters
    ----------
    ds : xr.Dataset
        Dataset containing at least u10 and v10 (10 m wind components).
    to_height : float
        Target height to scale wind speed to (m), typically the hub height.
    from_height : float
        Source height of provided winds (m), usually 10 m for ECMWF.
    wnd_shear_exp : float
        Wind shear exponent for the power-law wind profile (default ≈ 1/7).

    Returns
    -------
    xr.DataArray
        Area-averaged, hub-height wind speed over the region of interest.
    """

    # Select a spatial subset (Germany onshore box)
    subset = ds.sel(
        latitude=slice(55.0, 49.0),
        longitude=slice(6.0, 15.0)
    )

    # Compute 10 m wind speed magnitude from components
    subset['windspeed'] = np.sqrt(subset["u10"]**2 + subset["v10"]**2)

    # Scale wind speed to hub height using the power-law profile:
    # U(z) = U(z0) * (z / z0) ** alpha
    subset["windspeed"] = subset['windspeed'] * (to_height / from_height) ** wnd_shear_exp

    # Return spatial mean across latitude and longitude
    return subset['windspeed'].mean(dim=("latitude", "longitude"))


def get_capacity_facors(data: xr):
    """
    Convert wind speed (m/s) to capacity factor (0–1) using the turbine power curve.
    Note: function name has a minor typo; left unchanged for compatibility.

    Parameters
    ----------
    data : xr.DataArray
        Hub-height wind speed.

    Returns
    -------
    xr.DataArray
        Capacity factor with same dimensions as input (values between 0 and 1).
    """

    # Interpolation kernel: map wind speed -> normalized power (POW/P)
    def apply_power_curve(da):
        return np.interp(da, V, POW / P)  # normalized output (0–1)

    # Apply elementwise across the DataArray (supports Dask parallelization)
    cf = xr.apply_ufunc(
        apply_power_curve,
        data,
        input_core_dims=[[]],
        output_core_dims=[[]],
        output_dtypes=[data.dtype],
        dask="parallelized",
    )

    # Name the resulting variable
    cf.name = "capacity_factor"

    return cf


def get_pypsa_export_structure(data: xr):
    """
    Reshape capacity factor data into a list of pandas DataFrames for PyPSA export.
    Each DataFrame corresponds to one 'number' index (e.g., grid point or asset),
    with a fixed 3-hourly time index and linear interpolation for gaps.

    Parameters
    ----------
    data : xr.DataArray
        Capacity factor data with dimensions including 'number' and time-like
        coordinate accessible via 'valid_time' when iterating over chunks.

    Returns
    -------
    list[pd.DataFrame]
        One DataFrame per asset: column 'cf_onwind_{i+1}' indexed by 3-hourly timestamps.
    """

    cf = []

    # Iterate over assets indexed by the 'number' coordinate
    for i in range(0, data.number[-1].values):
        # Build the target regular time index (adjust end as needed)
        index = pd.date_range(
            start="2019-01-01 00:00",
            # CHANGE END HERE if your dataset extends further
            end="2020-01-02 00:00",
            freq="3h"
        )

        # Initialize empty series for this asset
        df = pd.DataFrame(index=index)
        df[f'cf_onwind_{i+1}'] = np.NaN

        # Fill DataFrame from each chunk in 'data'
        for data_chunk in data:
            dt = data_chunk['valid_time'].values
            dt_str = pd.to_datetime(dt).strftime("%Y-%m-%d %H:%M:%S")
            df.loc[dt_str, f'cf_onwind_{i+1}'] = data_chunk[i].values

        # Fill any missing timestamps by linear interpolation
        df[f'cf_onwind_{i+1}'] = df[f'cf_onwind_{i+1}'].interpolate(method="linear")
        cf.append(df)

    return cf


def prepare_dataset_wind():
    """
    End-to-end pipeline:
      1) Open all ECMWF GRIB files containing u10 (165) and v10 (166).
      2) Compute area-mean hub-height wind speed time series for the onshore box.
      3) Convert wind speed to capacity factor via the power curve.
      4) Reshape into PyPSA-ready DataFrames (3-hourly index).

    Returns
    -------
    list[pd.DataFrame]
        List of DataFrames with capacity factor time series per asset.
    """

    liste = []

    # Loop through all GRIB files in the specified directory
    for file_name in glob.glob("/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten/ECWMF_Data/*.grb"):
        ds = xr.open_dataset(
            file_name,
            engine="cfgrib",
            backend_kwargs={
                "filter_by_keys": {
                    "paramId": [165, 166],  # 165=u10, 166=v10
                }
            }
        )

        # Compute area-averaged hub-height wind speed
        ds = average_lat_long(ds)

        # Add a (string) time coordinate based on the dataset's time value
        ds.expand_dims(time=[str(ds['time'].values)[:10]])

        liste.append(ds)

    # Concatenate along time and sort chronologically
    data = xr.concat(liste, dim='time')
    data = data.sortby("time")

    # Map wind speed -> capacity factor (0–1)
    data = get_capacity_facors(data)

    # Build PyPSA export structure (list of DataFrames)
    data = get_pypsa_export_structure(data)

    return data


In [4]:
data = prepare_dataset_wind()

for i in range(0,50):
    data[i] = data[i].loc[:'2019-12-31 22:00']

In [5]:
list_onwind = []
for i in range(0,50):
    list_onwind.append((data[i].sum().max()))

print(np.mean(list_onwind))

709.464621597608


In [3]:
data = pd.read_pickle('/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten/cappacity_factors_prepared/cf_onwind.pkl')
pd.to_pickle(data, "/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten/cappacity_factors_prepared/cf_onwind.pkl")